# Student Workbook: Decision Trees

This is your copy of the class. Same data, same charts, same order -- but the core calculations are left for
you to fill in. Nothing here is graded. The point is simply to rebuild, in your own hands, what we did together
in class.

Each exercise looks like this:

### ✏️ Try it yourself
A short instruction.

followed by a blank code cell for you to fill in, and a collapsed **Show solution** block you can open if you
get stuck or want to check your work.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split

pd.set_option("display.precision", 3)

We left Session 3 with a straight decision boundary separating pass from fail. Let's check how well that boundary handles a trickier version of the passing rule.

In [2]:
pass_df = pd.read_csv("../data/exam_pass_3d.csv")
pass_df.head()

,hours_studied,practice_problems,sleep_hours,passed
0,7.0,9,8.5,1
1,6.6,18,6.0,0
2,0.7,1,8.1,0
3,7.0,16,4.8,0
4,3.2,20,7.2,0


### ✏️ Try it yourself

Fit a `LogisticRegression` on `hours_studied` and `practice_problems` to predict `passed`. Compute its accuracy on the full dataset, and compare it to the baseline accuracy of always guessing the majority class (`1 - pass_df["passed"].mean()`).

In [3]:
# TODO: fit LogisticRegression, then print its accuracy and the baseline accuracy

<details>
<summary>Show solution</summary>

```python
recap_model = LogisticRegression()
recap_model.fit(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])

recap_accuracy = recap_model.score(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
baseline_accuracy = 1 - pass_df["passed"].mean()

print(f"Logistic regression accuracy:          {recap_accuracy:.3f}")
print(f"Baseline (always guess the majority):  {baseline_accuracy:.3f}")
```

</details>

Continuing with the worked model so the rest of the notebook has something to build on:

In [4]:
recap_model = LogisticRegression()
recap_model.fit(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
recap_accuracy = recap_model.score(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
baseline_accuracy = 1 - pass_df["passed"].mean()
print(f"Logistic regression accuracy:          {recap_accuracy:.3f}")
print(f"Baseline (always guess the majority):  {baseline_accuracy:.3f}")

Logistic regression accuracy:          0.764
Baseline (always guess the majority):  0.727


Barely better than just guessing the majority class every time. Something about this data is fighting the model.

## Why Would a Straight Boundary Fail Here?

The true rule behind `exam_pass_3d.csv` isn't "more hours always help" or "more practice always helps." It's a
**sweet spot**: you need to study *enough*, and you need to practice a *moderate* amount -- too few practice
problems and you're underprepared, too many and it starts to look like guessing/rushing.

A straight decision boundary can express "more of this feature always pushes toward pass" or "always pushes
toward fail." It cannot express "there's a sweet spot in the middle" -- that shape needs at least one bend, and
a straight line has none.

> ### 💭 Think about it
>
> Can you think of other real-world rules that have a "too little OR too much" shape, where a straight-line rule wouldn't work?

## Thinking Like a Tree: The Twenty Questions Game

Before fitting anything, let's reason it out by hand -- the way you might play Twenty Questions.

1. **Did the student study at least 4 hours?** If no → predict **fail**.
2. If yes: **did they solve between 5 and 15 practice problems?** If yes → predict **pass**. If no → predict **fail**.

- **Root** — the first question asked of every row
- **Split** — any yes/no question that divides the data into two groups
- **Leaf** — a final box with no more questions, just a prediction

> ### 💭 Think about it
>
> If you're playing a number-guessing game (1-100) and can only ask yes/no questions, what's the fewest questions you need in the worst case? How does that connect to how deep a tree needs to be?

## Anatomy of a Tree

```text
                     hours_studied >= 4?
                    /                   \
                  no                    yes
                   |                     |
                 FAIL          practice_problems in [5, 15]?
                                /                        \
                              yes                        no
                               |                          |
                             PASS                       FAIL
```

## From Boundary Line to Boundary Staircase

First, a quick check against data you already know.

In [5]:
warmup_df = pd.read_csv("../data/exam_pass_2d.csv")
warmup_df.head()

,hours_studied,practice_problems,passed
0,6.7,12,1
1,6.3,6,1
2,1.7,7,0
3,1.5,8,0
4,6.4,3,1


### ✏️ Try it yourself

Fit both a `LogisticRegression` and a `DecisionTreeClassifier(max_depth=3)` on `warmup_df`'s two features, and print each model's accuracy on the full dataset.

In [6]:
# TODO: fit warmup_logistic and warmup_tree, then print both accuracies

<details>
<summary>Show solution</summary>

```python
warmup_logistic = LogisticRegression()
warmup_logistic.fit(warmup_df[["hours_studied", "practice_problems"]], warmup_df["passed"])

warmup_tree = DecisionTreeClassifier(max_depth=3, random_state=0)
warmup_tree.fit(warmup_df[["hours_studied", "practice_problems"]], warmup_df["passed"])

print(f"Logistic regression accuracy (Session 3 data): {warmup_logistic.score(warmup_df[['hours_studied', 'practice_problems']], warmup_df['passed']):.3f}")
print(f"Decision tree accuracy (Session 3 data):        {warmup_tree.score(warmup_df[['hours_studied', 'practice_problems']], warmup_df['passed']):.3f}")
```

</details>

Not a huge difference on that older dataset -- because that data's true boundary really was close to a straight line. Let's look at data where it isn't.

<details>
<summary>Show code</summary>

```python
tree_2feat_model = DecisionTreeClassifier(max_depth=3, random_state=0)
tree_2feat_model.fit(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])

logistic_2feat_model = LogisticRegression()
logistic_2feat_model.fit(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])

tree_accuracy = tree_2feat_model.score(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
logistic_accuracy = logistic_2feat_model.score(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
print(f"Logistic regression accuracy: {logistic_accuracy:.3f}")
print(f"Decision tree accuracy:       {tree_accuracy:.3f}")

grid_hours = np.linspace(pass_df["hours_studied"].min(), pass_df["hours_studied"].max(), 200)
grid_practice = np.linspace(pass_df["practice_problems"].min(), pass_df["practice_problems"].max(), 200)
grid_x, grid_y = np.meshgrid(grid_hours, grid_practice)
grid_points = pd.DataFrame({"hours_studied": grid_x.ravel(), "practice_problems": grid_y.ravel()})
grid_predictions = tree_2feat_model.predict(grid_points).reshape(grid_x.shape)

w1, w2 = logistic_2feat_model.coef_[0]
b = logistic_2feat_model.intercept_[0]
boundary_x1 = np.linspace(pass_df["hours_studied"].min(), pass_df["hours_studied"].max(), 50)
boundary_x2 = -(b + w1 * boundary_x1) / w2

boundary_figure = go.Figure()
boundary_figure.add_trace(
    go.Contour(
        x=grid_hours, y=grid_practice, z=grid_predictions,
        showscale=False, colorscale=[[0, "rgba(220,38,38,0.15)"], [1, "rgba(22,163,74,0.15)"]],
        contours=dict(start=0, end=1, size=1, coloring="fill"),
        line=dict(width=0), hoverinfo="skip",
    )
)
for passed_value, color, label in [(0, "#dc2626", "actual: failed"), (1, "#16a34a", "actual: passed")]:
    subset = pass_df[pass_df["passed"] == passed_value]
    boundary_figure.add_trace(
        go.Scatter(
            x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
            marker=dict(size=9, color=color, line=dict(width=1, color="white")), name=label,
        )
    )
boundary_figure.add_trace(
    go.Scatter(
        x=boundary_x1, y=boundary_x2, mode="lines",
        line=dict(color="#7c3aed", width=3, dash="dash"), name="logistic regression boundary",
    )
)
boundary_figure.update_layout(
    title="Decision Tree Region vs. Logistic Regression Boundary",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
boundary_figure.show()
```

</details>

In [7]:
tree_2feat_model = DecisionTreeClassifier(max_depth=3, random_state=0)
tree_2feat_model.fit(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])

logistic_2feat_model = LogisticRegression()
logistic_2feat_model.fit(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])

tree_accuracy = tree_2feat_model.score(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
logistic_accuracy = logistic_2feat_model.score(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
print(f"Logistic regression accuracy: {logistic_accuracy:.3f}")
print(f"Decision tree accuracy:       {tree_accuracy:.3f}")

grid_hours = np.linspace(pass_df["hours_studied"].min(), pass_df["hours_studied"].max(), 200)
grid_practice = np.linspace(pass_df["practice_problems"].min(), pass_df["practice_problems"].max(), 200)
grid_x, grid_y = np.meshgrid(grid_hours, grid_practice)
grid_points = pd.DataFrame({"hours_studied": grid_x.ravel(), "practice_problems": grid_y.ravel()})
grid_predictions = tree_2feat_model.predict(grid_points).reshape(grid_x.shape)

w1, w2 = logistic_2feat_model.coef_[0]
b = logistic_2feat_model.intercept_[0]
boundary_x1 = np.linspace(pass_df["hours_studied"].min(), pass_df["hours_studied"].max(), 50)
boundary_x2 = -(b + w1 * boundary_x1) / w2

boundary_figure = go.Figure()
boundary_figure.add_trace(
    go.Contour(
        x=grid_hours, y=grid_practice, z=grid_predictions,
        showscale=False, colorscale=[[0, "rgba(220,38,38,0.15)"], [1, "rgba(22,163,74,0.15)"]],
        contours=dict(start=0, end=1, size=1, coloring="fill"),
        line=dict(width=0), hoverinfo="skip",
    )
)
for passed_value, color, label in [(0, "#dc2626", "actual: failed"), (1, "#16a34a", "actual: passed")]:
    subset = pass_df[pass_df["passed"] == passed_value]
    boundary_figure.add_trace(
        go.Scatter(
            x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
            marker=dict(size=9, color=color, line=dict(width=1, color="white")), name=label,
        )
    )
boundary_figure.add_trace(
    go.Scatter(
        x=boundary_x1, y=boundary_x2, mode="lines",
        line=dict(color="#7c3aed", width=3, dash="dash"), name="logistic regression boundary",
    )
)
boundary_figure.update_layout(
    title="Decision Tree Region vs. Logistic Regression Boundary",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
boundary_figure.show()

Logistic regression accuracy: 0.764
Decision tree accuracy:       0.982


The tree's shaded region wraps around the "sweet spot" pass zone -- something a single straight line geometrically cannot do.

## Training Data vs. New Data: A Quick Detour

So far we've graded every model on the same data it learned from. Trees are flexible enough to potentially just
*memorize* the training rows rather than learn the real pattern. To catch that, we hold out a slice of data the
model never sees while training, and grade it only on that slice.

### ✏️ Try it yourself

Split `pass_df`'s two features (`X`) and target (`y`) into `X_train, X_test, y_train, y_test` using `train_test_split` with `test_size=0.3, random_state=0, stratify=y`.

In [8]:
X = pass_df[["hours_studied", "practice_problems"]]
y = pass_df["passed"]
# TODO: split X, y into X_train, X_test, y_train, y_test

<details>
<summary>Show solution</summary>

```python
X = pass_df[["hours_studied", "practice_problems"]]
y = pass_df["passed"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")
```

</details>

Continuing with the worked split:

In [9]:
X = pass_df[["hours_studied", "practice_problems"]]
y = pass_df["passed"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

## Tree Depth: Underfitting, Overfitting, and the Sweet Spot

`max_depth` limits how many questions a tree is allowed to ask before it must land on a leaf.

### ✏️ Try it yourself

For `max_depth` in `[1, 3, None]`: fit a `DecisionTreeClassifier` on `X_train, y_train`, and record its train accuracy and test accuracy. Put the results in a DataFrame.

In [10]:
depth_results = []
for depth in [1, 3, None]:
    # TODO: fit a DecisionTreeClassifier(max_depth=depth), then append a dict with
    #       "max_depth", "train_accuracy", "test_accuracy" to depth_results
    pass

pd.DataFrame(depth_results)

""


<details>
<summary>Show solution</summary>

```python
depth_results = []
for depth in [1, 3, None]:
    depth_model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    depth_model.fit(X_train, y_train)
    depth_results.append({
        "max_depth": depth if depth is not None else "None (unlimited)",
        "train_accuracy": depth_model.score(X_train, y_train),
        "test_accuracy": depth_model.score(X_test, y_test),
    })

pd.DataFrame(depth_results)
```

</details>

Continuing with the worked results:

In [11]:
depth_results = []
for depth in [1, 3, None]:
    depth_model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    depth_model.fit(X_train, y_train)
    depth_results.append({
        "max_depth": depth if depth is not None else "None (unlimited)",
        "train_accuracy": depth_model.score(X_train, y_train),
        "test_accuracy": depth_model.score(X_test, y_test),
    })
pd.DataFrame(depth_results)

,max_depth,train_accuracy,test_accuracy
0,1,0.779,0.697
1,3,0.974,0.939
2,None (unlimited),1.000,0.879


> ### 🧑‍🏫 Instructor note
>
> "More depth" feels like it should mean "more accurate" -- and on the TRAIN column it does, right up to a
> perfect 1.000. But look at the TEST column: the depth=3 tree beats the unlimited-depth tree there. The
> unlimited tree memorized quirks of the training rows that don't generalize to new students -- that's overfitting.

## Reading the Tree Itself

No fancy diagram needed for this -- scikit-learn can print the tree's actual questions as text.

<details>
<summary>Show code</summary>

```python
depth3_model = DecisionTreeClassifier(max_depth=3, random_state=0)
depth3_model.fit(X_train, y_train)
print(export_text(depth3_model, feature_names=["hours_studied", "practice_problems"]))
```

</details>

In [12]:
depth3_model = DecisionTreeClassifier(max_depth=3, random_state=0)
depth3_model.fit(X_train, y_train)
print(export_text(depth3_model, feature_names=["hours_studied", "practice_problems"]))

|--- hours_studied <= 5.45
|   |--- hours_studied <= 4.20
|   |   |--- class: 0
|   |--- hours_studied >  4.20
|   |   |--- hours_studied <= 4.45
|   |   |   |--- class: 1
|   |   |--- hours_studied >  4.45
|   |   |   |--- class: 0
|--- hours_studied >  5.45
|   |--- practice_problems <= 4.50
|   |   |--- class: 0
|   |--- practice_problems >  4.50
|   |   |--- practice_problems <= 15.50
|   |   |   |--- class: 1
|   |   |--- practice_problems >  15.50
|   |   |   |--- class: 0



Compare this to the two questions we wrote by hand earlier -- the fitted tree found nearly the same rule on its own, just from the data.

## What a Split Is Actually Optimizing

$$\text{Gini} = 1 - \sum_i p_i^2$$

You don't need to memorize this. Just like Session 1's SSE, it's simply the number the algorithm is trying to
minimize every time it chooses a split: lower Gini after a split means the two resulting groups are more "pure.

## Feature Importance: A Preview of Session 5

`exam_pass_3d.csv` has a third column, `sleep_hours`, that we haven't used yet -- and it's deliberately
unrelated to whether a student passes. Let's see if the tree notices.

### ✏️ Try it yourself

Fit a `DecisionTreeClassifier(max_depth=3)` on all three features (`hours_studied`, `practice_problems`, `sleep_hours`), then make a bar chart of `.feature_importances_`.

In [13]:
X3 = pass_df[["hours_studied", "practice_problems", "sleep_hours"]]
y3 = pass_df["passed"]

# TODO: fit importance_model, then build a go.Bar chart of its feature_importances_

<details>
<summary>Show solution</summary>

```python
X3 = pass_df[["hours_studied", "practice_problems", "sleep_hours"]]
y3 = pass_df["passed"]

importance_model = DecisionTreeClassifier(max_depth=3, random_state=0)
importance_model.fit(X3, y3)

importance_figure = go.Figure()
importance_figure.add_trace(
    go.Bar(x=list(X3.columns), y=importance_model.feature_importances_, marker=dict(color="#f59e0b"))
)
importance_figure.update_layout(
    title="Feature Importance (single tree)",
    xaxis_title="feature", yaxis_title="importance",
    template="plotly_white", width=650, height=450,
)
importance_figure.show()
```

</details>

`sleep_hours` should land at essentially zero importance -- the tree never found a split on it worth making. This is the tree's own version of Session 2's p-values: a way of unmasking a feature that looks like data but carries no signal.

> ### 📝 Note
>
> The exact importance numbers can shift slightly depending on how you built `importance_model` above -- what matters is that `sleep_hours` is far smaller than the other two, not the exact digits.

## What We Covered Today

- A straight decision boundary can't represent a "sweet spot" (too little OR too much) rule.
- Decision trees ask a sequence of yes/no questions (splits), starting from a root, ending in leaves.
- `max_depth` controls how many questions the tree is allowed to ask -- too few underfits, too many overfits.
- `train_test_split` lets us measure overfitting by grading a model on data it never trained on.
- `export_text()` shows the tree's actual learned questions.
- `feature_importances_` reveals which features the tree actually used -- including unmasking a useless one.

## Final Check

1. Why can't a straight decision boundary represent the "sweet spot" rule in `exam_pass_3d.csv`?
2. A tree gets 100% train accuracy but only 75% test accuracy. What's going on, and which knob would you turn to fix it?
3. Why might a feature with **zero** importance in a fitted tree still be a perfectly good column to have collected?

<details>
<summary>Show solution</summary>

1. A straight boundary can only express "more of this feature always helps" or "always hurts" -- it has no way
   to bend back on itself to carve out a middle region.
2. The tree has overfit -- it memorized quirks of the training rows instead of the general pattern. Reducing
   `max_depth` (or otherwise constraining the tree) usually helps.
3. Zero importance just means *this* tree, on *this* data, found no useful split on it -- not that the feature
   is inherently useless. It could matter for a different target, a different dataset, or simply confirm (as
   here) that a variable really is noise.

</details>